<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260317_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.makedirs("/content/project/templates", exist_ok=True)

In [2]:
%%writefile /content/project/server.js

const express = require('express');
const path = require('path');
const app = express();
const port = 3000;

const templatePath = path.join(__dirname, 'templates');

// 메인 페이지 라우트: index.html 파일을 클라이언트에 전송
app.get('/', (req, res) => {
    res.sendFile(path.join(templatePath, 'index.html'));
});

// [예제 1] 단일 Route Parameter를 활용한 유저 정보 API
app.get('/api/user/:name', (req, res) => {
    res.json({ name: req.params.name });
});

// [예제 2] 다중 Route Parameters를 활용한 배송 정보 API
app.get('/api/delivery/:sender/:receiver', (req, res) => {
    res.json({
        sender: req.params.sender,
        receiver: req.params.receiver
    });
});

// [예제 3] Route Parameter 값 검증 및 조건부 데이터 API
app.get('/api/id-check/:number', (req, res) => {
    const number = req.params.number;
    const isNumeric = /^\d+$/.test(number);
    res.json({
        id: number,
        isValid: isNumeric
    });
});

// [예제 4] 정적 경로와 변수 경로의 우선순위 확인용 API
app.get('/api/member/admin', (req, res) => {
    res.json({ name: '관리자', role: '시스템 제어' });
});

app.get('/api/member/:name', (req, res) => {
    res.json({ name: req.params.name, role: '일반 회원' });
});

// [예제 5] Route Parameters와 Query Strings 혼용 API
app.get('/api/search/:category', (req, res) => {
    res.json({
        category: req.params.category,
        keyword: req.query.keyword
    });
});

app.listen(port, () => {
    console.log(`서버 실행 중: http://localhost:${port}`);
});

Writing /content/project/server.js


In [3]:
%%writefile /content/project/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>Express Routing & fetch 실습</title>
    <style>
        .example-box { border: 1px solid #ccc; padding: 10px; margin-bottom: 10px; }
        .result { color: blue; font-weight: bold; }
    </style>
</head>
<body>
    <h1>Express Routing 및 fetch API 실습</h1>

    <div class="example-box">
        <h3>1. 단일 Route Parameter (이름 확인)</h3>
        <button onclick="getUser('홍길동')">홍길동 조회</button>
        <p>결과: <span id="res1" class="result"></span></p>
    </div>

    <div class="example-box">
        <h3>2. 다중 Route Parameters (배송 확인)</h3>
        <button onclick="getDelivery('김철수', '이영희')">배송 조회</button>
        <p>결과: <span id="res2" class="result"></span></p>
    </div>

    <div class="example-box">
        <h3>3. 정규식 검증 (숫자 확인)</h3>
        <button onclick="checkId('2026')">숫자 입력(2026)</button>
        <button onclick="checkId('abc')">문자 입력(abc)</button>
        <p>결과: <span id="res3" class="result"></span></p>
    </div>

    <div class="example-box">
        <h3>4. 경로 우선순위 확인</h3>
        <button onclick="getMember('admin')">admin 요청</button>
        <button onclick="getMember('박지민')">박지민 요청</button>
        <p>결과: <span id="res4" class="result"></span></p>
    </div>

    <div class="example-box">
        <h3>5. Route Parameters + Query Strings</h3>
        <button onclick="searchItem('전자제품', '노트북')">전자제품/노트북 검색</button>
        <p>결과: <span id="res5" class="result"></span></p>
    </div>

    <script>
        // 예제 1
        function getUser(name) {
            fetch('/api/user/' + name)
                .then(res => res.json())
                .then(data => document.getElementById('res1').innerText = data.name + '님 접속');
        }

        // 예제 2
        function getDelivery(sender, receiver) {
            fetch(`/api/delivery/${sender}/${receiver}`)
                .then(res => res.json())
                .then(data => document.getElementById('res2').innerText = `${data.sender} -> ${data.receiver}`);
        }

        // 예제 3
        function checkId(num) {
            fetch('/api/id-check/' + num)
                .then(res => res.json())
                .then(data => {
                    const status = data.isValid ? '유효한 번호' : '잘못된 형식';
                    document.getElementById('res3').innerText = `${data.id} : ${status}`;
                });
        }

        // 예제 4
        function getMember(name) {
            fetch('/api/member/' + name)
                .then(res => res.json())
                .then(data => document.getElementById('res4').innerText = `${data.name} (${data.role})`);
        }

        // 예제 5
        function searchItem(cat, word) {
            fetch(`/api/search/${cat}?keyword=${word}`)
                .then(res => res.json())
                .then(data => document.getElementById('res5').innerText = `${data.category}에서 '${data.keyword}' 검색`);
        }
    </script>
</body>
</html>

Writing /content/project/templates/index.html


In [4]:
%%bash
cd /content/project
npm install express mysql2 bcrypt
npm install -g cloudflared


added 79 packages in 7s

24 packages are looking for funding
  run `npm fund` for details

added 1 package in 2s


In [5]:
%%bash
cd /content/project
nohup node server.js > /content/project/server.log 2>&1 &
sleep 2
cat /content/project/server.log

서버 실행 중: http://localhost:3000


In [6]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &
sleep 5
cat /content/tunnel.log | grep trycloudflare.com

2026-03-17T07:26:25Z INF Requesting new quick Tunnel on trycloudflare.com...


In [7]:
%%bash
cat /content/tunnel.log | grep trycloudflare.com

2026-03-17T07:26:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-17T07:26:33Z INF |  https://behaviour-season-vatican-jamie.trycloudflare.com                                  |
